# DocuDevs: Fill a PDF Template with AcroForm Fields

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/docudevs/python-examples/blob/main/07-pdf-template.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/docudevs/python-examples)

This notebook uses the public `docs/authorization-to-disclose-health-information.pdf` sample. It is already a fillable PDF, so the workflow is just the PDF template workflow:

1. Upload the PDF as a template.
2. Inspect the discovered AcroForm fields.
3. Fill selected text and checkbox fields.
4. Save and verify the filled PDF.


In [ ]:
# Install dependencies (required in Colab)
# Colab includes packages such as gradio and google-genai with their own
# pydantic/httpx constraints. Install compatible shared deps first, then
# install the SDK without letting older SDK metadata downgrade them.
%pip install -q --upgrade "pydantic>=2.0,<=2.12.3" "httpx>=0.28.1,<1.0" "attrs>=21.3.0" "python-dateutil>=2.8.0" "click>=8.0.0"
%pip install -q --upgrade --no-deps docu-devs-api-client


## Setup

These examples use the published `docu-devs-api-client` package from PyPI. For Colab, store `DOCUDEVS_API_KEY` in the 🔑 Secrets sidebar. You can also set it as an environment variable before running the notebook.


In [ ]:
import importlib.metadata
import io
import json
import os
import urllib.request
from http import HTTPStatus
from pathlib import Path
from typing import Any, Optional

from pypdf import PdfReader

from docudevs import DocuDevsClient, TemplateFillRequest

# Set your API key:
# Option 1: Colab Secrets (recommended) — add DOCUDEVS_API_KEY in the 🔑 sidebar
# Option 2: Environment variable
# Option 3: Set directly below
try:
    from google.colab import userdata
    API_KEY = userdata.get("DOCUDEVS_API_KEY")
except Exception:
    API_KEY = os.getenv("DOCUDEVS_API_KEY", "your-api-key-here")

if not API_KEY or API_KEY == "your-api-key-here":
    raise ValueError("Set your DOCUDEVS_API_KEY - get one at https://docudevs.ai")

client = DocuDevsClient(token=API_KEY)
Path("outputs").mkdir(exist_ok=True)
print(f"Connected to DocuDevs API with SDK {importlib.metadata.version('docu-devs-api-client')}")


## Load the Public Fillable PDF


In [2]:
docs_dir = Path("docs")
docs_dir.mkdir(exist_ok=True)

template_path = docs_dir / "authorization-to-disclose-health-information.pdf"
template_url = (
    "https://raw.githubusercontent.com/docudevs/python-examples/main/docs/"
    "authorization-to-disclose-health-information.pdf"
)

if not template_path.exists():
    urllib.request.urlretrieve(template_url, str(template_path))

if not template_path.exists():
    raise FileNotFoundError(f"Missing {template_path}")

template_bytes = template_path.read_bytes()
template_id = "authorization_to_disclose_health_information"

local_reader = PdfReader(io.BytesIO(template_bytes))
local_fields = local_reader.get_fields() or {}

print(f"Loaded {template_path.name}: {len(template_bytes):,} bytes")
print(f"Pages: {len(local_reader.pages)}")
print(f"Native AcroForm fields: {len(local_fields)}")
print("First local field names:")
for index, name in enumerate(list(local_fields)[:12], start=1):
    field = local_fields[name]
    print(f"{index:02d}. {name} | type={field.get('/FT')}")


Loaded authorization-to-disclose-health-information.pdf: 249,883 bytes
Pages: 3
Native AcroForm fields: 30
First local field names:
01. Patient Name | type=/Tx
02. Medical Record Number | type=/Tx
03. Birth Date | type=/Tx
04. Email | type=/Tx
05. Recipient Name | type=/Tx
06. Address | type=/Tx
07. City | type=/Tx
08. State | type=/Tx
09. Zip Code | type=/Tx
10. Phone | type=/Tx
11. undefined | type=/Tx
12. Email_2 | type=/Tx


## Upload the PDF Template and Read Metadata


In [4]:
def to_plain(value: Any) -> Any:
    if hasattr(value, "to_dict"):
        return to_plain(value.to_dict())
    if hasattr(value, "model_dump"):
        return to_plain(value.model_dump())
    if isinstance(value, list):
        return [to_plain(item) for item in value]
    if isinstance(value, dict):
        return {key: to_plain(item) for key, item in value.items()}
    return value


def response_payload(response: Any) -> Any:
    parsed = getattr(response, "parsed", None)
    if parsed is not None:
        return to_plain(parsed)
    content = getattr(response, "content", None)
    if isinstance(content, (bytes, bytearray)) and content:
        text = bytes(content).decode("utf-8", errors="replace")
        try:
            return json.loads(text)
        except Exception:
            return text
    return None


def require_ok(response: Any, operation: str) -> None:
    status_code = getattr(response, "status_code", None)
    if status_code == HTTPStatus.OK:
        return
    content = getattr(response, "content", b"")
    if isinstance(content, (bytes, bytearray)):
        content = bytes(content).decode("utf-8", errors="replace")
    raise RuntimeError(f"{operation} failed with status {status_code}: {content}")


upload_response = await client.upload_template(
    name=template_id,
    document=io.BytesIO(template_bytes),
    file_name=template_path.name,
    mime_type="application/pdf",
)
require_ok(upload_response, "template upload")
print(f"Uploaded template {template_id}: status {upload_response.status_code}")

upload_payload = response_payload(upload_response)
template_fields = []

if isinstance(upload_payload, dict):
    candidate_fields = upload_payload.get("formFields")
    if isinstance(candidate_fields, list) and candidate_fields:
        template_fields = candidate_fields

if not template_fields:
    template_fields = await client.wait_for_template_metadata(
        template_id,
        timeout=60,
        poll_interval=1,
    )

print(f"Template fields discovered by API: {len(template_fields)}")
for index, field in enumerate(template_fields, start=1):
    print(
        f"{index:02d}. {field.get('name')} | "
        f"type={field.get('type')} | value={field.get('value')!r} | "
        f"options={field.get('options', [])}"
    )


Uploaded template authorization_to_disclose_health_information: status 200
Template fields discovered by API: 30
01. Patient Name | type=TEXT | value=None | options=[]
02. Medical Record Number | type=TEXT | value=None | options=[]
03. Birth Date | type=TEXT | value=None | options=[]
04. Email | type=TEXT | value=None | options=[]
05. Recipient Name | type=TEXT | value=None | options=[]
06. Address | type=TEXT | value=None | options=[]
07. City | type=TEXT | value=None | options=[]
08. State | type=TEXT | value=None | options=[]
09. Zip Code | type=TEXT | value=None | options=[]
10. Phone | type=TEXT | value=None | options=[]
11. undefined | type=TEXT | value=None | options=[]
12. Email_2 | type=TEXT | value=None | options=[]
13. Legal | type=PUSH_BUTTON | value='OTHER' | options=[]
14. Insurance | type=PUSH_BUTTON | value='OTHER' | options=[]
15. Medical Certification | type=PUSH_BUTTON | value='OTHER' | options=[]
16. Other | type=PUSH_BUTTON | value='OTHER' | options=[]
17. Form Com

## Fill Selected Fields


In [5]:
sample_values_by_name: dict[str, Any] = {
    "Patient Name": "Jordan Example",
    "Medical Record Number": "MRN-123456",
    "Birth Date": "1988-04-12",
    "Email": "jordan.example@example.com",
    "Recipient Name": "Northside Clinic",
    "Address": "123 Health Ave",
    "City": "Seattle",
    "State": "WA",
    "Zip Code": "98101",
    "Phone": "206-555-0199",
    "Email_2": "records@example-clinic.test",
    "Date": "2026-06-03",
    "Legal": True,
    "Insurance": True,
    "Medical Records": True,
    "Diagnostic Images": True,
}


def sample_value_for_field(field: dict[str, Any]) -> Optional[Any]:
    name = field.get("name")
    if name in sample_values_by_name:
        return sample_values_by_name[name]
    return None


fill_fields = {
    field["name"]: value
    for field in template_fields
    for value in [sample_value_for_field(field)]
    if field.get("name") and value is not None
}

print(f"Fields to fill: {len(fill_fields)}")
print(json.dumps(fill_fields, indent=2, ensure_ascii=False))


Fields to fill: 16
{
  "Patient Name": "Jordan Example",
  "Medical Record Number": "MRN-123456",
  "Birth Date": "1988-04-12",
  "Email": "jordan.example@example.com",
  "Recipient Name": "Northside Clinic",
  "Address": "123 Health Ave",
  "City": "Seattle",
  "State": "WA",
  "Zip Code": "98101",
  "Phone": "206-555-0199",
  "Email_2": "records@example-clinic.test",
  "Legal": true,
  "Insurance": true,
  "Medical Records": true,
  "Diagnostic Images": true,
  "Date": "2026-06-03"
}


In [6]:
fill_response = await client.fill_with_retry(
    name=template_id,
    body=TemplateFillRequest(fields=fill_fields),
    timeout=60,
    poll_interval=2,
)
require_ok(fill_response, "template fill")

filled_pdf_bytes = bytes(fill_response.content)
filled_pdf_path = Path("outputs") / f"{template_id}-filled.pdf"
filled_pdf_path.write_bytes(filled_pdf_bytes)

print(f"Filled PDF saved to {filled_pdf_path}")
print(f"Size: {len(filled_pdf_bytes):,} bytes")


Filled PDF saved to outputs/authorization_to_disclose_health_information-filled.pdf
Size: 265,174 bytes


## Verify the Filled PDF Locally


In [ ]:
filled_reader = PdfReader(str(filled_pdf_path))
filled_fields = filled_reader.get_fields() or {}

for name, expected in fill_fields.items():
    actual = filled_fields.get(name, {}).get("/V")
    print(f"{name}: {actual!r} (expected {expected!r})")
